# GNN-ReGVD+ — обучение в Google Colab

Ноутбук рассчитан на **бесплатный тариф (T4, 16 ГБ)**.

Важное перед стартом: после исправления LoRA появился режим `contextual`, в котором
GraphCodeBERT реально исполняется на каждом батче. Он даёт работающую LoRA, но
на порядок дороже прежнего. Поэтому здесь два профиля:

| Профиль | `--encoder_mode` | LoRA обучается | Батч на T4 | ~Время эпохи | Зачем |
|---|---|---|---|---|---|
| baseline | `static` | нет | 128 | ~2 мин | честный ReGVD-baseline, дёшево |
| lora | `contextual` | **да** | 24–32 + AMP | ~10–12 мин | основной эксперимент |

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, platform
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu  :", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
import multiprocessing; print("cpus :", multiprocessing.cpu_count())

## 1. Google Drive

Сессия Colab обрывается — веса должны лежать на Drive, иначе всё сгорит.
`run.py` теперь пишет `checkpoint-last` после каждой эпохи и умеет с него продолжать.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/gnn-regvd'
os.makedirs(DRIVE_DIR, exist_ok=True)
print("checkpoints ->", DRIVE_DIR)

## 2. Код и зависимости

In [ ]:
%cd /content
# вариант A — свой репозиторий
BRANCH = "fix/lora-gradient-flow-and-module0"   # ветка с исправлениями; main их ещё не содержит
!git clone -b {BRANCH} https://github.com/deccersw/GNN-ReGVD-Plus.git 2>/dev/null || echo "already cloned"
!cd GNN-ReGVD-Plus && git log --oneline -1
# вариант B — распаковать архив, залитый на Drive:
# !unzip -q -o /content/drive/MyDrive/GNN-ReGVD-Plus.zip -d /content
%cd /content/GNN-ReGVD-Plus
!ls

In [ ]:
# Colab уже несёт torch/sklearn/scipy. Явно нужны только эти.
!pip install -q "transformers>=4.30" faiss-cpu
# apex НЕ нужен: смешанная точность идёт через torch.amp
import transformers; print("transformers:", transformers.__version__)

## 3. Проверка перед долгим запуском

Двадцать секунд, которые экономят часы. Убеждаемся, что каждый обучаемый тензор
действительно в графе автограда — именно эта проверка ловит класс ошибок, из-за
которого LoRA раньше молча не обучалась.

In [ ]:
%cd /content/GNN-ReGVD-Plus/code
import torch, logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
from transformers import RobertaConfig, RobertaForSequenceClassification, RobertaTokenizer
from model import GNNReGVD, check_gradient_flow

cfg = RobertaConfig.from_pretrained("microsoft/graphcodebert-base"); cfg.num_labels = 1
tok = RobertaTokenizer.from_pretrained("microsoft/graphcodebert-base")
enc = RobertaForSequenceClassification.from_pretrained("microsoft/graphcodebert-base", config=cfg)

args = type("A", (), dict(
    gnn="ReGCN", format="uni", window_size=5, block_size=400,
    feature_dim_size=768, hidden_size=128, num_GNN_layers=2,
    remove_residual=False, att_op="mul", num_classes=1,
    use_lora=True, lora_rank=8, lora_alpha=16,
    use_faiss=True, embed_dim=512, encoder_mode="auto"))()

model = GNNReGVD(enc, cfg, tok, args).cuda()
print("encoder_mode:", model.encoder_mode)
print(model.get_trainable_params_info())

ids = torch.randint(5, 50000, (4, 400)).cuda()
lab = torch.tensor([1., 0., 1., 0.]).cuda()

# Проверять надо той же целевой функцией, что и в обучении: FAISS-голова
# учится контрастивным членом, и по одному классификационному лоссу она
# выглядит «вне графа» — ложная тревога.
from losses import SupervisedContrastiveLoss
contrastive = SupervisedContrastiveLoss(temperature=0.07)
objective = lambda out, y: 0.7 * out[0] + 0.3 * contrastive(out[2], y)

rep = check_gradient_flow(model, ids, lab, loss_fn=objective)

assert rep["detached"] == [], f"вне графа автограда: {rep['detached'][:3]}"
print("\nOK: все обучаемые тензоры в графе автограда, "
      f"LoRA среди них: {len([n for n in rep['with_grad'] + rep['zero_grad'] if 'lora_' in n])}")

## 4. Подбор батча под свою карту

T4 выдают разные — лучше померить, чем угадать. Ячейка ищет наибольший батч,
который влезает в память, и печатает скорость.

In [ ]:
import torch, time, gc

def try_batch(model, bs, seq=400, amp=True, steps=3):
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=amp)
    ids = torch.randint(5, 50000, (bs, seq)).cuda()
    lab = (torch.rand(bs) > .5).float().cuda()
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
    for _ in range(steps):
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=amp):
            loss = model(ids, lab)[0]
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    torch.cuda.synchronize()
    dt = (time.time() - t0) / steps
    return torch.cuda.max_memory_allocated()/2**30, bs/dt

best = None
for bs in (8, 16, 24, 32, 48, 64):
    try:
        mem, sps = try_batch(model, bs)
        print(f"batch={bs:>3}  peak={mem:5.2f} GiB  {sps:5.1f} samples/s")
        best = bs
    except torch.cuda.OutOfMemoryError:
        print(f"batch={bs:>3}  OOM"); break
    finally:
        gc.collect(); torch.cuda.empty_cache()
print("\nрекомендуемый --train_batch_size:", best)

## 5. Обучение

Заметное отличие от README: `--epoch 100` в Colab недостижимо для `contextual`
(это ~18 часов). Ставим 25 и опираемся на возобновление.

`--num_workers 2` — на бесплатном тарифе всего 2 vCPU, а граф строится на CPU
внутри `forward()`, так что лишние воркеры только мешают.

Чекпоинты пишутся в формате `slim` (по умолчанию): 5 МБ вместо 478 МБ,
потому что замороженные веса GraphCodeBERT восстанавливаются из
`--model_name_or_path`. За 25 эпох на Drive уйдёт ~125 МБ вместо ~12 ГБ.

`--early_stopping_patience 5` остановит прогон, когда `eval_loss` перестанет
улучшаться пять эпох подряд. На baseline валидация развернулась на 10-й
эпохе, так что это экономит часы.


In [ ]:
%cd /content/GNN-ReGVD-Plus/code
import os
OUT = '/content/drive/MyDrive/gnn-regvd/lora_contextual'
os.makedirs(OUT, exist_ok=True)

BATCH = 24        # из ячейки 4
EPOCHS = 25

!python run.py \
  --output_dir={OUT} \
  --model_type=roberta \
  --tokenizer_name=microsoft/graphcodebert-base \
  --model_name_or_path=microsoft/graphcodebert-base \
  --do_train --do_eval --evaluate_during_training \
  --train_data_file=../dataset/train.jsonl \
  --eval_data_file=../dataset/valid.jsonl \
  --test_data_file=../dataset/test.jsonl \
  --block_size 400 \
  --train_batch_size {BATCH} --eval_batch_size {BATCH} \
  --gradient_accumulation_steps 4 \
  --num_workers 2 --fp16 \
  --gnn ReGCN --format uni --window_size 5 \
  --hidden_size 128 --num_GNN_layers 2 --num_classes 1 \
  --use_lora --lora_rank 8 --lora_alpha 16 --encoder_mode contextual \
  --use_faiss --embed_dim 512 \
  --contrastive_weight 0.3 --contrastive_loss supcon \
  --learning_rate 5e-4 --epoch {EPOCHS} --seed 42 \
  --early_stopping_patience 5 --early_stopping_metric eval_loss 2>&1 | tail -60

### Обрыв сессии

Просто запустите ячейку 5 заново **с тем же `--output_dir`**. `run.py` найдёт
`checkpoint-last`, восстановит веса, оптимизатор, планировщик, номер эпохи и
лучшую точность, и продолжит с места остановки.

Проверить, что снимок на месте:

In [ ]:
import json, os
last = os.path.join(OUT, 'checkpoint-last')
print("файлы:", sorted(os.listdir(last)) if os.path.isdir(last) else "снимка ещё нет")
if os.path.isdir(last):
    print("состояние:", json.load(open(os.path.join(last, 'training_state.json'))))
    print("архитектура:", json.load(open(os.path.join(last, 'model_config.json')))['encoder_mode'])

## 6. Baseline без LoRA

Нужен для сравнения: он показывает, что даёт именно адаптация, а не всё остальное.
Дёшев, потому что трансформер не исполняется.

In [ ]:
OUT_BASE = '/content/drive/MyDrive/gnn-regvd/baseline_static'
!python run.py \
  --output_dir={OUT_BASE} \
  --model_type=roberta \
  --tokenizer_name=microsoft/graphcodebert-base \
  --model_name_or_path=microsoft/graphcodebert-base \
  --do_train --do_eval --evaluate_during_training \
  --train_data_file=../dataset/train.jsonl \
  --eval_data_file=../dataset/valid.jsonl \
  --block_size 400 --train_batch_size 128 --eval_batch_size 128 \
  --num_workers 2 \
  --gnn ReGCN --format uni --window_size 5 \
  --hidden_size 128 --num_GNN_layers 2 --num_classes 1 \
  --encoder_mode static \
  --use_faiss --embed_dim 512 \
  --contrastive_weight 0.3 --contrastive_loss supcon \
  --learning_rate 5e-4 --epoch 50 --seed 42 \
  --early_stopping_patience 5 --early_stopping_metric eval_loss 2>&1 | tail -40

## 7. Тест и метрики

In [ ]:
!python run.py \
  --output_dir={OUT} \
  --model_type=roberta \
  --tokenizer_name=microsoft/graphcodebert-base \
  --model_name_or_path=microsoft/graphcodebert-base \
  --do_test \
  --train_data_file=../dataset/train.jsonl \
  --eval_data_file=../dataset/valid.jsonl \
  --test_data_file=../dataset/test.jsonl \
  --block_size 400 --eval_batch_size 24 --num_workers 2 \
  --gnn ReGCN --format uni --window_size 5 \
  --hidden_size 128 --num_GNN_layers 2 --num_classes 1 \
  --use_lora --encoder_mode contextual --use_faiss --embed_dim 512 2>&1 | tail -20

%cd /content/GNN-ReGVD-Plus
!python evaluate.py --mode gnn \
  --model-path {OUT}/checkpoint-best-acc/model.bin \
  --faiss-dir {OUT}/faiss_index 2>&1 | tail -20

## 8. Сканирование проекта (Module 0)

Межпроцедурный инлайнинг работает на CPU и модели не требует — можно проверить
отдельно, ещё до окончания обучения.

In [ ]:
%cd /content/GNN-ReGVD-Plus
!python scan_cli.py --project test_samples/projects/p01_simple \
    --build-units-only --inline-depth 2

# с обученной моделью:
# !python scan_cli.py --project /content/myproject --inline-depth 2 \
#     --model-path {OUT}/checkpoint-best-acc/model.bin \
#     --faiss-dir {OUT}/faiss_index

## Если что-то пошло не так

**CUDA out of memory** — уменьшите `--train_batch_size` и поднимите
`--gradient_accumulation_steps`, чтобы эффективный батч не менялся. Проверьте,
что `--fp16` включён.

**Эпоха идёт очень долго при низкой загрузке GPU** — это ожидаемо: граф строится
на CPU внутри `forward()` (~2–4 мс на образец) и блокирует GPU. На бесплатном
тарифе с 2 vCPU это заметная доля времени эпохи.

**`ImportError: cannot import name 'AdamW'`** — не должно происходить, импорты
защищены; если всё же видите, значит запускается старая копия кода.

**Модель обучается, но LoRA не меняется** — проверьте `encoder_mode` в логе.
Должно быть `contextual`. При `static` трансформер не исполняется и адаптеры
не получают градиента.

**После возобновления метрики скачут** — убедитесь, что `--output_dir` тот же:
при другом каталоге `checkpoint-last` не находится и обучение стартует заново.